# Ingestion and table-aware retrieval

Many FinanceBench questions have a table cell as their answer, and a chunker that treats a filing as running text can cut a cell away from the column header and row label that give it meaning. This notebook follows one path through RI #3, the third reference implementation (RI): how a filing becomes section folders, why splitting a table matters, and what changes on FinanceBench's 150 questions when the chunker keeps a table with its heading and repeats the header rows when it has to split.

The expensive steps, converting 84 PDFs and embedding about 95,000 chunks, sit behind `RUN_EXPENSIVE`. With the flag off, the notebook reads the results committed in `outputs/`, so it needs neither the filings nor a model.

In [1]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path

from aieng.advanced_rag.cache.paths import root as cache_root
from aieng.advanced_rag.corpora.financebench import PAGE_BASE, load_queries
from aieng.advanced_rag.retrieval.chunkers import all_facts
from aieng.advanced_rag.retrieval.filings import chunk_filing, load_document
from aieng.advanced_rag.retrieval.tables import attachment_report, contains_value


# Converting the filings took about half an hour split across three processes, and
# scoring took about 75 minutes. Leave this False to read the committed results.
RUN_EXPENSIVE = False
# Print question, evidence, and filing text from your local copy. FinanceBench is
# licensed CC BY-NC 4.0, so the committed outputs carry identifiers and metrics only.
SHOW_CORPUS_TEXT = False

K = 10
HERE = Path.cwd()
REPO = HERE.parents[1]
OUTPUTS = HERE / "outputs"

queries = load_queries()
kinds = {}
for query in queries:
    kinds[query.category] = kinds.get(query.category, 0) + 1
print(f"{len(queries)} questions over {len({q.doc_id for q in queries})} filings")
for kind, count in sorted(kinds.items()):
    print(f"  {kind}: {count}")

example = next(q for q in queries if q.reasoning == "Numerical reasoning")
print()
print(f"example: {example.query_id} on {example.doc_id} ({example.reasoning})")
print(f"gold pages (zero-based, page {PAGE_BASE} is the first): {example.gold_pages}")
if SHOW_CORPUS_TEXT:
    print("Q:", example.question[:200])
    print("A:", example.answer)
    print("evidence:", example.evidence_texts[0][:160].replace("\n", " "), "...")

150 questions over 84 filings
  domain-relevant: 50
  metrics-generated: 50
  novel-generated: 50

example: financebench_id_02987 on ACTIVISIONBLIZZARD_2019_10K (Numerical reasoning)
gold pages (zero-based, page 0 is the first): (68, 69)


## Gold evidence is a page

FinanceBench names the pages that hold each answer, counting from zero. Retrieval judgments follow: a chunk is relevant when it covers a gold page. Docling counts pages from one, so the package converts once, in `chunk_pages`, and every page you see from here is zero-based.

In [2]:
# A two-page filing small enough to read, from the package's test fixtures. Its
# company, labels, and figures are invented, so its text prints by default.
fixture = REPO / "aieng-advanced-rag/tests/aieng/advanced_rag/ingestion/fixtures"
folder = Path(tempfile.mkdtemp())
(folder / "document.json").write_bytes((fixture / "acme-10k.docling.json").read_bytes())
document = load_document(folder)


def holds(chunk, fact):
    """Whether a chunk holds a cell's table row, with its label and value."""
    start, end = fact.table_span
    in_table = chunk.start < end and start < chunk.end
    return in_table and f"| {fact.row_label} |" in chunk.text and contains_value(chunk.text, fact.value)


def split_cell(filing):
    """Return the first numeric cell a chunk holds without its column header."""
    for fact in all_facts(filing.items):
        for chunk in filing.chunks:
            if holds(chunk, fact) and not all(part in chunk.text for part in fact.header_parts):
                return fact
    return None


chunkings = {s: chunk_filing("ACME", document, strategy=s, max_chars=160) for s in ("plain", "table_aware")}
cell = split_cell(chunkings["plain"])
print(f"cell: row {cell.row}, column {cell.column}, header {cell.header_parts}")
print()
for strategy, filing in chunkings.items():
    held = [c for c in filing.chunks if holds(c, cell)]
    report = attachment_report(filing.chunks, all_facts(filing.items))
    print(
        f"--- {strategy}: {len(filing.chunks)} chunks, "
        f"{report['attachment_rate']:.0%} of cell appearances keep header and label"
    )
    print(held[0].text)
    print("has the period header:", cell.header_parts[0] in held[0].text)
    print()

cell: row 3, column 1, header ('December 31, 2022', '(In millions)')

--- plain: 5 chunks, 44% of cell appearances keep header and label
021 |
|  | (In millions) | (In millions) |
| Net revenue: |  |  |
| Instruments | 4,812 | 2,957 |
| Retail | 5,318 | 4,406 |

Results of operations

The discuss
has the period header: False

--- table_aware: 7 chunks, 100% of cell appearances keep header and label
Segment results
|  | December 31, 2022 | December 31, 2021 |
|  | (In millions) | (In millions) |
| Net revenue: |  |  |
| Instruments | 4,812 | 2,957 |
has the period header: True



At 160 characters the plain chunker lands the `Instruments | 4,812` row in a chunk with no column header, which leaves a reader with a bare number and no year. The table-aware chunker keeps the table with its section heading and repeats the header rows when it has to split.

The same measurement on a filing from the corpus loads it by its identifier from the section folders that `ingest_financebench.py` writes to your local cache. The chunks are cut at 1,200 characters, the size the scored comparison below uses. The chunk text prints only when `SHOW_CORPUS_TEXT` is on.

In [3]:
FILING = "AMD_2022_10K"
filing_folder = Path(cache_root()) / "ingestion" / "financebench" / FILING

if not (filing_folder / "document.json").exists():
    print(f"{FILING} has no section folder yet. Run ingest_financebench.py to build it (see the README).")
else:
    corpus_document = load_document(filing_folder)
    corpus_chunkings = {s: chunk_filing(FILING, corpus_document, strategy=s) for s in ("plain", "table_aware")}
    for strategy, filing in corpus_chunkings.items():
        report = attachment_report(filing.chunks, all_facts(filing.items))
        print(
            f"{FILING} {strategy}: {len(filing.chunks):,} chunks, {report['attachment_rate']:.1%} of "
            f"cell appearances keep header and label, {report['cells_never_complete']:,} of "
            f"{report['scored_cells']:,} cells never complete"
        )
    corpus_cell = split_cell(corpus_chunkings["plain"])
    if corpus_cell is not None:
        start, end = corpus_cell.table_span
        print(
            f"first split cell: row {corpus_cell.row}, column {corpus_cell.column} of the table at characters {start:,} to {end:,}"
        )
        if SHOW_CORPUS_TEXT:
            for strategy, filing in corpus_chunkings.items():
                print(f"--- {strategy}")
                print(next(c for c in filing.chunks if holds(c, corpus_cell)).text)

AMD_2022_10K plain: 420 chunks, 56.2% of cell appearances keep header and label, 366 of 1,064 cells never complete
AMD_2022_10K table_aware: 519 chunks, 97.7% of cell appearances keep header and label, 0 of 1,064 cells never complete
first split cell: row 11, column 1 of the table at characters 213,491 to 214,138


## A filing as a folder

The ingestion harness writes one folder per filing: the converter's `document.json`, an `ingestion.json` manifest, and one markdown file per page, named from page zero. Each parse is keyed by the source file's digest and the parser version, so a changed parser rebuilds and a repeat run is a cache hit.

In [4]:
if RUN_EXPENSIVE:
    subprocess.run([sys.executable, str(HERE / "ingest_financebench.py")], check=True)
    subprocess.run(
        [sys.executable, str(HERE / "ingest_financebench.py"), "--summarise", str(OUTPUTS / "ingestion_summary.json")],
        check=True,
    )

summary = json.loads((OUTPUTS / "ingestion_summary.json").read_text())
print(
    f"{summary['documents']} filings, {summary['pages']:,} pages with content, "
    f"{summary['tables']:,} tables, {summary['failures']} failures"
)
largest = sorted(summary["per_document"].items(), key=lambda kv: -kv[1]["tables"])[:3]
for name, counts in largest:
    print(f"  {name}: {counts['pages']} pages, {counts['tables']} tables")

84 filings, 11,678 pages with content, 7,600 tables, 0 failures
  JPMORGAN_2022_10K: 374 pages, 292 tables
  JPMORGAN_2023Q2_10Q: 214 pages, 262 tables
  JPMORGAN_2022Q2_10Q: 195 pages, 241 tables


## Two chunkings, two retrievers

Both chunkings cut the same serialised text at 1,200 characters. The plain one slices overlapping windows. The table-aware one keeps a table with its heading and repeats the header rows when it splits. BM25 is a keyword-ranking function and needs no model. The dense arm is `BAAI/bge-small-en-v1.5` run locally. Retrieval is scoped to the question's own filing, as it is for contracts in RI #1, and judged at `k=10` against the gold pages. The table reports retrieval failure, the share of questions with no relevant chunk in the top 10, and mean reciprocal rank (MRR).

In [5]:
if RUN_EXPENSIVE:
    subprocess.run(
        [
            sys.executable,
            str(HERE / "score_table_retrieval.py"),
            "--force-refresh",
            "--out",
            str(OUTPUTS / "table_retrieval_scores.json"),
        ],
        check=True,
    )

scores = json.loads((OUTPUTS / "table_retrieval_scores.json").read_text())
print("| chunking | chunks | arm | failure | MRR |")
print("| --- | ---: | --- | ---: | ---: |")
for strategy in ("plain", "table_aware"):
    row = scores[strategy]
    for arm, report in row["retrieval"].items():
        print(
            f"| {strategy} | {row['chunks']:,} | {arm} | {report['retrieval_failure_rate']:.4f} | {report['mrr']:.4f} |"
        )
print()
for strategy in ("plain", "table_aware"):
    cells = scores[strategy]["attachment"]
    print(
        f"{strategy}: {cells['attachment_rate']:.1%} of chunk-cell pairs keep header and "
        f"label, {cells['cell_recoverable_rate']:.1%} of {cells['scored_cells']:,} cells "
        f"appear complete in some chunk"
    )

| chunking | chunks | arm | failure | MRR |
| --- | ---: | --- | ---: | ---: |
| plain | 41,401 | bm25 | 0.6267 | 0.1965 |
| plain | 41,401 | dense | 0.3400 | 0.3459 |
| table_aware | 53,715 | bm25 | 0.6600 | 0.1677 |
| table_aware | 53,715 | dense | 0.4000 | 0.3542 |

plain: 51.3% of chunk-cell pairs keep header and label, 62.5% of 189,040 cells appear complete in some chunk
table_aware: 94.6% of chunk-cell pairs keep header and label, 99.9% of 189,040 cells appear complete in some chunk


## Reading the result

Across the 84 filings, a numeric cell sits in a chunk with its column header and row label in 94.6% of its appearances with table-aware chunking, against 51.3% for plain windows. Of the 189,040 scored numeric cells, 99.9% appear complete in some chunk, against 62.5%.

Retrieval does not improve on these questions. With the dense arm, retrieval failure rises from 0.34 to 0.40 and MRR stays near 0.35. BM25 is worse on both measures. Three caveats apply:

- The 150 questions are scored once, and 0.06 is nine questions. Nothing here tests whether that difference is more than noise. The paired comparison is a separate step.
- The table-aware chunking makes 53,715 chunks against 41,401, so each gold page holds more, smaller chunks. Failure marks a question as failed when no relevant chunk reaches the top 10, so it compares across chunkings. Recall divides by the number of relevant chunks, which differs between the two.
- Page-level judgments reward finding the right page and say nothing about whether the chunk that reaches the reader holds the cell. Attachment measures that, and it improves.

## What to change next

To try another retriever, add it to the `arms` dictionary in `score_table_retrieval.py` and re-run it. To see how chunk size moves both measures, pass `--max-chars`. The table storage setting decides whether a table is kept as markdown, as an image, or as both, and comparing its values on these 150 questions is the next comparison.